# Da Capo, Run C

This notebook runs Run C of [Da Capo](https://github.com/ninondev/DaCapo) on ACE-Step 1.5 (turbo). Run C replicates B2 of Run B on 96 new
lines and melodies, with B2's two recipes fixed in advance (addendum 4):

- S*: the Voice Oohs guide for the first 3 of 8 steps, with the lyric clock at beta 8 on the seven alignment heads.
- S*_b0: the same guide without the clock, plus two controls (the clock on random heads, the clock with its
  windows rotated).

It asks whether the clock helps at all (R1: the bootstrap 2.5th percentile of the gap in J is above 0) and whether
the capability holds (R2: J(S*) of at least 0.50, 48 of 96 melodies). Every decision rule is applied in code, so the
run needs no one watching it.

1. Choose Runtime > Change runtime type > L4 GPU, on the default runtime version. The notebook stops on another
   GPU, because the registration and the time projection assume an L4.
2. Choose Runtime > Run all and allow Google Drive access. Drive keeps the ledgers, audio, results and stage
   status in `MyDrive/dacapo_runC` (about 5 GB; setup stops if less than 6.1 GB is free). Without Drive the
   notebook stops before running, because a runtime reset would lose the run.
3. The run is expected to take about 6 hours on one L4 (1,536 runs at Run B's measured 12.1 s each, plus install,
   setup, A0 and the decision). If the runtime resets or Colab stops it, choose Run all again: it resumes from
   Drive at the commit it started with and never generates a finished run twice. Resume on an L4 with the same
   runtime version: generation refuses a GPU or software stack other than the one A0 ran on. If the run stops
   before it is done, the outcome cell says how far it got; Run all goes on from there.
4. The last cell downloads `dacapo_runC_*.zip` (results, ledgers, logs and the 192-clip demo) and, as separate
   files, the blind A/B pack `c1_blind_pack.zip` and its key `c1_blind_key.json`. Give listeners the pack only,
   before they hear the demo.

The code is the registered commit (`0c97266`),
ACE-Step is pinned to `ca1e85f` and the weights to revision `19671f4`.

In [ ]:
SCALE = "full"                          # "full" is the registered run; "tiny" only tests the code
DRIVE_DIR = "/content/drive/MyDrive/dacapo_runC"
ALLOW_NO_DRIVE = False                  # True runs without Drive: a runtime reset then loses the run
ALLOW_OTHER_PYTHON = False              # True runs on a Python other than 3.12 or 3.13 (the pins were tested there)
ALLOW_OTHER_GPU = False                 # True runs on a GPU other than an NVIDIA L4 (the registration assumes an L4)
ALLOW_RUNTIME_CHANGE = False            # True lets a resumed run generate on a GPU or software stack other than A0's,
                                        # with the change recorded in every row
REPO_COMMIT = None                      # None = the commit this scale's run recorded on Drive, else the registered one
REGISTERED = '0c972666c9e4f8f14001cf700365f587e0f0569b'  # None = the commit that last changed addendum 4 on the default branch
import os, platform, re, shutil, subprocess, sys
print("Python", platform.python_version())
if ".".join(platform.python_version_tuple()[:2]) not in ('3.12', '3.13') and not ALLOW_OTHER_PYTHON:
    raise SystemExit("Run C runs on Colab's default runtime with Python 3.12 or 3.13: change the runtime version")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", gpu or "none")
if not re.search(r"\bL4\b", gpu) and not ALLOW_OTHER_GPU:
    raise SystemExit(f"Run C runs on an NVIDIA L4, not on {gpu or 'a runtime without a GPU'}: choose Runtime > Change "
                     "runtime type > L4 GPU, or set ALLOW_OTHER_GPU = True")

In [ ]:
# Google Drive keeps the run state, so Run all can resume after a runtime reset
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("run state in", DRIVE_DIR)
except Exception as exc:
    from IPython.display import HTML, display
    display(HTML('<h1 style="color:#c00">GOOGLE DRIVE IS NOT MOUNTED</h1>'
                 '<h2>The run state would stay on this runtime and be lost if it resets.</h2>'
                 '<h2>Mount Drive and choose Run all again, or set ALLOW_NO_DRIVE = True in the first cell.</h2>'))
    print("#" * 60, "\nGOOGLE DRIVE IS NOT MOUNTED:", exc, "\n" + "#" * 60)
    if not ALLOW_NO_DRIVE:
        raise SystemExit("Run C stops: Drive is not mounted")
    DRIVE_DIR = None

In [ ]:
# Code: the registered commit. A resumed run checks out the commit its first runtime used (kept on Drive for this
# scale only). A run has started once the C1 ledger holds a run; a pin left by a run that never started must match
# the registered commit. Set REPO_COMMIT to move to another commit on purpose.
def git(*a):
    return subprocess.run(["git", "-C", "/content/work/repo", *a], check=True, capture_output=True, text=True).stdout.strip()
def has_runs(ledger):
    return os.path.exists(ledger) and any('"run_id"' in line for line in open(ledger))
if not os.path.exists("/content/work/repo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/ninondev/DaCapo", "/content/work/repo"], check=True)
git("fetch", "-q", "origin")
registered = REGISTERED or git("log", "-1", "--format=%H", "origin/HEAD", "--", "experiments/e5_motif/registration/addendum_4.json")
if not registered:
    raise SystemExit("addendum 4 is not on the default branch: Run C is not registered")
state_dir = f"{DRIVE_DIR}/{SCALE}" if DRIVE_DIR else None
commit_file = f"{state_dir}/repo_commit.txt" if state_dir else None
started = bool(state_dir) and has_runs(f"{state_dir}/c1/test/ledger.jsonl")
if not REPO_COMMIT and commit_file and os.path.exists(commit_file):
    pin = open(commit_file).read().strip()
    if pin and (started or pin == registered):
        REPO_COMMIT = pin
    elif pin:
        raise SystemExit(f"Drive pins commit {pin[:7]} for scale {SCALE}, but no {SCALE} run has generated anything and "
                         f"the registered commit is {registered[:7]}. Delete {commit_file} to use the registered commit, "
                         f"or set REPO_COMMIT.")
want = REPO_COMMIT or registered
git("checkout", "-q", want)
head = git("rev-parse", "HEAD")
if head != git("rev-parse", want):
    raise SystemExit(f"HEAD is {head[:7]}, not the pinned commit {want[:7]}")
if commit_file:
    os.makedirs(state_dir, exist_ok=True)
    open(commit_file, "w").write(head + "\n")
print("Da Capo HEAD", head, "| registered", registered[:7], "| resumed run" if started else "| new run")
!git -C /content/work/repo log -1 --format="Da Capo HEAD %h %cd"
!git -C /content/work/repo log -1 --format="addendum 4 last changed in %h %cd" -- experiments/e5_motif/registration/addendum_4.json

In [ ]:
# Dependencies: FluidSynth and a General MIDI soundfont, ACE-Step inference, the pinned measurement tools (as Run A
# and Run B). Colab's PyTorch stays as it is.
torch_before = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
!apt-get -qq update > /dev/null && apt-get -qq install -y fluidsynth fluid-soundfont-gm ffmpeg > /dev/null
!pip -q install transformers==4.57.6 diffusers==0.39.0 accelerate==1.15.0 vector-quantize-pytorch einops loguru \
     soundfile scipy safetensors huggingface_hub mido torchfcpe resampy scikit-learn nvidia-ml-py swift-f0==0.3.0 onnxruntime==1.31.0 librosa==1.0.0 soxr==1.1.0 mir_eval==0.8.2 pretty_midi==0.2.11.post0 demucs==4.1.0
!pip -q install --no-deps basic-pitch==0.4.0
torch_after = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
assert torch_after == torch_before, f"pip changed torch from {torch_before} to {torch_after}"
if not os.path.exists("/content/work/ACE-Step-1.5"):
    !git clone -q https://github.com/ace-step/ACE-Step-1.5 /content/work/ACE-Step-1.5
!cd /content/work/ACE-Step-1.5 && git checkout -q ca1e85fe9430179831e6bc6be790c332190a3866 && git log -1 --format="ACE-Step %h %cd"
!fluidsynth --version | head -1
print("torch", torch_after)

In [ ]:
# Weights at the registered revision: turbo DiT, VAE, text encoder (ACE-Step's loader also checks for the 1.7B LM)
from huggingface_hub import snapshot_download
snapshot_download("ACE-Step/Ace-Step1.5", revision="19671f406d603126926c1b7e2adc169acbcade22", local_dir="/content/work/ace_ckpt",
                  allow_patterns=["acestep-v15-turbo/*", "vae/*", "Qwen3-Embedding-0.6B/*", "acestep-5Hz-lm-1.7B/*", "config.json"])
print("ok")

In [ ]:
# The registrations: addendum 2 (E5 v2 and E6), the B1 code addendum, addendum 3 (B2) and addendum 4 (Run C) must all
# be current. DISABLE_SAFETENSORS_CONVERSION: the CTC model loads pytorch_model.bin at its pinned revision; this stops
# transformers from also fetching a converted copy in the background
env = dict(os.environ, ACESTEP_REPO="/content/work/ACE-Step-1.5", ACESTEP_CHECKPOINTS_DIR="/content/work/ace_ckpt",
           PYTHONPATH="/content/work/repo", DACAPO_SF2="/usr/share/sounds/sf2/FluidR3_GM.sf2",
           CUBLAS_WORKSPACE_CONFIG=":4096:8", ACESTEP_VAE_DECODE_CHUNK_SIZE="256", DISABLE_SAFETENSORS_CONVERSION="true",
           DACAPO_ALLOW_RUNTIME_CHANGE="1" if ALLOW_RUNTIME_CHANGE else "0", DACAPO_DIE_WITH_PARENT="1")
bad = []
for script in ("experiments/e5_motif/register.py", "experiments/e6_clock/register_b1code.py",
               "experiments/e8_clock_sing/register_b2.py", "experiments/e9_clock_replicate/register_c1.py"):
    r = subprocess.run([sys.executable, f"/content/work/repo/{script}", "--check"], env=env, capture_output=True, text=True)
    print(script, "->", (r.stdout + r.stderr).strip()[-300:])
    bad += [script] if r.returncode else []
if bad:
    raise SystemExit(f"registration check failed: {bad}")

In [ ]:
# Run C. Every stage writes its log; the outcome is decided by the registered rule in code.
args = [sys.executable, "/content/work/repo/experiments/run_c.py", "--scale", SCALE, "--device", "cuda"]
if DRIVE_DIR:
    args += ["--state", DRIVE_DIR]
p = subprocess.Popen(args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in p.stdout:
        print(line, end="")
    p.wait()
except KeyboardInterrupt:   # a stopped cell stops the run too, so no second run writes to the same Drive folder
    p.terminate()
    p.wait()
    raise
print("run_c exit code", p.returncode)

In [ ]:
# Outcome summary
import json
root = f"{DRIVE_DIR}/{SCALE}" if DRIVE_DIR else ("/content/work/runC" if SCALE == "full" else f"/content/work/runC_{SCALE}")
status = json.load(open(f"{root}/status.json"))
for name, st in status.items():
    if name != "_meta":
        print(f"{name:8s}", st.get("state"), {k: st[k] for k in ("verdict", "reasons", "labels", "J", "blocks_done",
                                                                 "projection") if st.get(k) is not None})
if status.get("_meta", {}).get("resume"):
    print("#" * 60, "\n" + status["_meta"]["resume"], "\n" + "#" * 60)

In [ ]:
# Package and download; the blind A/B pack and its key are separate files
import shutil, time
name = f"/content/dacapo_runC_{time.strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(name, "zip", f"{root}/package")
print(name + ".zip", round(os.path.getsize(name + ".zip") / 1e6, 1), "MB")
from google.colab import files
files.download(name + ".zip")
for extra in ("c1_blind_pack.zip", "c1_blind_key.json"):
    if os.path.exists(f"{root}/{extra}"):
        files.download(f"{root}/{extra}")